In [1]:
# ================================
# GERMAN CREDIT MODEL (WITH SMOTE)
# ================================

# --- 1. Mount Drive ---
from google.colab import drive
drive.mount('/content/gdrive')

# --- 2. Imports ---
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- 3. Paths ---
data_dir = Path("/content/gdrive/MyDrive/Colab Notebooks/AIGerman")
out_dir = data_dir / "german_credit_outputs"
out_dir.mkdir(exist_ok=True)

# --- 4. Load Data ---
df = pd.read_csv(data_dir / "credit_g.csv")

# --- 5. Prepare Data ---
y = df["class"].map({"good": 0, "bad": 1})  # bad = 1 (important)
X = df.drop(columns=["class"])

cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# --- 6. Models WITH SMOTE ---
lr = ImbPipeline([
    ("prep", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

rf = ImbPipeline([
    ("prep", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

ensemble = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf)],
    voting="soft"
)

models = {
    "Logistic Regression + SMOTE": lr,
    "Random Forest + SMOTE": rf,
    "Soft Voting Ensemble": ensemble
}

# --- 7. Cross-validation (10-fold) ---
cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model, X, y,
        cv=10,
        scoring=["accuracy", "precision", "recall", "f1"],
        n_jobs=-1
    )

    cv_results.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean()
    })

cv_results = pd.DataFrame(cv_results)
print("\n=== CROSS VALIDATION RESULTS ===")
display(cv_results.round(4))

# --- 8. Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- 9. Train Final Model ---
final_model = models["Soft Voting Ensemble"]
final_model.fit(X_train, y_train)

prob = final_model.predict_proba(X_test)[:, 1]

# --- 10. Threshold Optimization ---
thresholds = np.linspace(0.1, 0.9, 50)

rows = []
best_threshold = 0
best_score = 0

for t in thresholds:
    preds = (prob >= t).astype(int)
    f1 = f1_score(y_test, preds)

    rows.append({"threshold": t, "f1": f1})

    if f1 > best_score:
        best_score = f1
        best_threshold = t

threshold_table = pd.DataFrame(rows)

print("\nSelected threshold:", round(best_threshold, 3))
display(threshold_table.head(10).round(4))

# --- 11. Final Evaluation ---
final_preds = (prob >= best_threshold).astype(int)

test_results = pd.DataFrame([{
    "accuracy": accuracy_score(y_test, final_preds),
    "precision": precision_score(y_test, final_preds),
    "recall": recall_score(y_test, final_preds),
    "f1": f1_score(y_test, final_preds)
}])

print("\n=== FINAL TEST RESULTS ===")
display(test_results.round(4))

# --- 12. ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(out_dir / "roc_curve.png")
plt.close()

# --- 13. Save Outputs ---
cv_results.to_csv(out_dir / "cv_results.csv", index=False)
threshold_table.to_csv(out_dir / "threshold_search.csv", index=False)
test_results.to_csv(out_dir / "test_results.csv", index=False)

# --- 14. Metadata ---
metadata = {
    "rows": int(len(X)),
    "n_features": int(X.shape[1]),
    "selected_threshold": float(best_threshold),
    "positive_class": "bad credit risk",
    "method": "SMOTE + Ensemble"
}

with open(out_dir / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved files:")
print(sorted(p.name for p in out_dir.iterdir()))

ModuleNotFoundError: No module named 'google.colab'